In [ ]:
from __future__ import annotations
import json
from typing import Optional
from core.services.comptabilite import comptabilite  # adapte l'import à ton arbre

from core.variables.variable import MAX_KEYWORDS,EN_COURS,EN_PAUSE,TERMINE
from core.models.step import Step
from core.models.objectif import Objective

In [ ]:
class LinkResult:
    """
    Retourné par link_transaction pour informer l'appelant de ce qui s'est passé.

    attributed  : {step_name: montant_attribué}
    overflow    : montant qui dépasse tous les steps disponibles
    overflowed  : True si l'objectif est dépassé dans sa globalité
    """
    def __init__(self):
        self.attributed : dict[str, float] = {}
        self.overflow   : float = 0.0
        self.overflowed : bool  = False

    def __repr__(self):
        return (f"LinkResult(attributed={self.attributed}, "
                f"overflow={self.overflow}, overflowed={self.overflowed})")

In [ ]:
from __future__ import annotations
import json
import pandas as pd
from datetime import datetime, date
from typing import Optional
from core.variables.variable import COLUMNS_STRUCTURE,_STOPWORDS



class ProjetManagement:
    """
    Cerveau du système de gestion de projets financiers.

    Responsabilités
    ---------------
    - Gérer N objectifs
    - Lier des transactions (issues de `comptabilite`) à des étapes
    - Redistribuer automatiquement le surplus vers la prochaine étape EN_COURS
    - Recommander des transactions similaires via keywords (manuels + auto)
    - Calculer les priorités (importance × retard × urgence × activité)
    - Sauvegarder / charger l'état en JSON
    """

    def __init__(self, compta, json_path: str):
        self.compta    = compta
        self.objectifs : list[Objective] = []
        self._load_goals(json_path)

    # ── Persistence ───────────────────────────────────────────────────────────

    def _load_goals(self, json_path: str) -> None:
        """Charge le fichier JSON au démarrage. Silencieux si absent."""
        import os
        self.json_path = json_path
        if not os.path.exists(self.json_path):
            return
        with open(self.json_path, encoding="utf-8") as f:
            self.objectifs = [Objective.from_dict(d) for d in json.load(f)]

    def save(self, path: str = None) -> None:
        """Sauvegarde — par défaut vers le json_path d'origine."""
        target = path or self.json_path
        with open(target, "w", encoding="utf-8") as f:
            json.dump([o.to_dict() for o in self.objectifs],
                      f, ensure_ascii=False, indent=2)

    # ── Fabrique rapide ───────────────────────────────────────────────────────

    def create_objective(self, name: str, but: str = "", importance: int = 1,
                         deadline: str = None, logo: str = "") -> Objective:
        """Crée et enregistre un nouvel objectif vide."""
        obj = Objective(name=name, but=but, importance=importance,
                        deadline=deadline, logo=logo)
        self.objectifs.append(obj)
        return obj

    def create_step(self, obj_name: str, name: str, target: float,
                    but: str = "", deadline: str = None,
                    keywords: list[str] = None,
                    max_keywords: int = MAX_KEYWORDS) -> Optional[Step]:
        """Crée et ajoute une étape à un objectif existant."""
        obj = self.get_objective(obj_name)
        if obj is None:
            return None
        step = Step(name=name, target=target, but=but,
                    deadline=deadline, keywords=keywords,
                    max_keywords=max_keywords)
        obj.add_step(step)
        return step

    # ── Accès ─────────────────────────────────────────────────────────────────

    def get_objective(self, name: str) -> Optional[Objective]:
        return next((o for o in self.objectifs if o.name == name), None)

    def _get_step(self, obj_name: str, step_name: str) -> Optional[Step]:
        obj = self.get_objective(obj_name)
        if obj is None:
            return None
        return next((s for s in obj.steps if s.name == step_name), None)

    def _get_transaction(self, account_name: str, tid: str) -> Optional[dict]:
        """Cherche une transaction par real_index dans un compte."""
        account = next((c for c in self.compta.liste_compte
                        if c.account_name == account_name), None)
        if account is None:
            return None
        row = account.df[account.df["real_index"] == tid]
        return row.iloc[0].to_dict() if not row.empty else None

    # ── Liaison transactions ──────────────────────────────────────────────────

    def link_transaction(self, obj_name: str, step_name: str,
                         account_name: str, tid: str,
                         is_expense: bool = False) -> LinkResult:
        """
        Attache une transaction à une étape.

        - Récupère le montant réel depuis le compte.
        - Remplit l'étape cible ; si surplus → cascade vers la prochaine étape EN_COURS.
        - Enrichit automatiquement les keywords de chaque étape touchée.
        - Retourne un LinkResult décrivant la distribution.
        """
        result = LinkResult()
        obj    = self.get_objective(obj_name)
        step   = self._get_step(obj_name, step_name)
        if obj is None or step is None:
            return result

        tx = self._get_transaction(account_name, tid)
        if tx is None:
            return result

        montant      = float(tx.get(COLUMNS_STRUCTURE[5], 0))
        current_step = step
        remaining    = montant

        while current_step is not None and remaining > 0:
            space = current_step.remaining_to_find

            if space <= 0:
                current_step = obj.next_open_step(current_step)
                continue

            absorbed   = min(remaining, space)
            remaining -= absorbed

            if is_expense:
                current_step.expenses_only += absorbed
                links = current_step.expense_links
            else:
                links = current_step.income_links

            current_step.current_total += absorbed

            links.setdefault(account_name, [])
            if tid not in links[account_name]:
                links[account_name].append(tid)

            self._enrich_keywords(current_step, tx)

            result.attributed[current_step.name] = (
                result.attributed.get(current_step.name, 0) + absorbed
            )

            if current_step.is_complete:
                current_step.status = TERMINE

            if remaining > 0:
                current_step = obj.next_open_step(current_step)

        if remaining > 0:
            result.overflow   = remaining
            result.overflowed = True

        return result

    def unlink_transaction(self, tid: str) -> bool:
        """Détache un ID de transaction partout (revenus et dépenses)."""
        found = False
        for obj in self.objectifs:
            for step in obj.steps:
                for links in (step.income_links, step.expense_links):
                    for account, ids in list(links.items()):
                        if tid in ids:
                            ids.remove(tid)
                            found = True
                            if not ids:
                                del links[account]
        return found

    # ── Recommandations ───────────────────────────────────────────────────────

    def _enrich_keywords(self, step: Step, tx: dict) -> None:
        """
        Extrait automatiquement des mots-clés depuis une transaction
        et les ajoute aux keywords de l'étape (sans doublon, en minuscule).
        Colonnes exploitées : Intitule, Categorie, Classe.
        Limite à step.max_keywords en gardant les plus récents.
        """
        sources = [
            tx.get(COLUMNS_STRUCTURE[1], ""),  # Intitule
            tx.get(COLUMNS_STRUCTURE[2], ""),  # Categorie
            tx.get(COLUMNS_STRUCTURE[3], ""),  # Classe
        ]
        for raw in sources:
            for word in str(raw).lower().split():
                if (len(word) > 2
                        and word not in _STOPWORDS
                        and word not in step.keywords):
                    step.keywords.append(word)

        if len(step.keywords) > step.max_keywords:
            step.keywords = step.keywords[-step.max_keywords:]

    def recommend_transactions(self, obj_name: str, step_name: str,
                                account_name: str,
                                max_results: int = 10) -> list[dict]:
        """
        Retourne les transactions du compte les plus susceptibles
        d'appartenir à cette étape, triées par score décroissant.
        Score = nb de keywords matchés dans (Intitule + Categorie + Classe).
        """
        step    = self._get_step(obj_name, step_name)
        account = next((c for c in self.compta.liste_compte
                        if c.account_name == account_name), None)
        if step is None or account is None or not step.keywords:
            return []

        keywords = [k.lower() for k in step.keywords]

        already_linked = set(
            tid
            for links in (step.income_links, step.expense_links)
            for ids in links.values()
            for tid in ids
        )

        results = []
        for _, row in account.df.iterrows():
            tid = row.get("real_index", "")
            if tid in already_linked:
                continue

            haystack = " ".join([
                str(row.get(COLUMNS_STRUCTURE[1], "")),
                str(row.get(COLUMNS_STRUCTURE[2], "")),
                str(row.get(COLUMNS_STRUCTURE[3], "")),
            ]).lower()

            score = sum(1 for kw in keywords if kw in haystack)
            if score > 0:
                results.append({"score": score, "transaction": row.to_dict()})

        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:max_results]

    # ── Priorités ─────────────────────────────────────────────────────────────

    def _compute_priority_score(self, obj: Objective, step: Step,
                                 window_months: int = 3) -> float:
        """
        score = importance × retard × urgence × activité

        retard   = 1 - (current / target)
        urgence  = 1 + (1 / mois_restants)       linéaire
        activité = 1 + bonus_régularité - malus_abandon
        """
        retard = 1 - (step.current_total / step.target) if step.target else 0

        urgence      = 1.0
        deadline_str = step.deadline or obj.deadline
        if deadline_str:
            try:
                deadline_date = datetime.strptime(deadline_str, "%Y-%m-%d").date()
                mois_restants = max(
                    (deadline_date - date.today()).days / 30.44,
                    0.1
                )
                urgence = 1 + (1 / mois_restants)
            except ValueError:
                pass

        activite = self._compute_activity_factor(step, window_months)

        return obj.importance * retard * urgence * activite

    def _compute_activity_factor(self, step: Step, window_months: int) -> float:
        """
        Analyse les transactions liées à l'étape sur les N derniers mois.

        Bonus  (+0.5 max) : investi régulièrement → score monte
        Malus  (-0.5 max) : rien depuis N mois    → score baisse
        Neutre (1.0)      : pas assez de données
        """
        cutoff       = datetime.now().timestamp() - (window_months * 30.44 * 86400)
        total_recent = 0.0
        nb_tx_recent = 0
        has_any_tx   = False

        all_links = {
            **{acc: ids for acc, ids in step.income_links.items()},
            **{acc: ids for acc, ids in step.expense_links.items()},
        }

        for account_name, tids in all_links.items():
            account = next((c for c in self.compta.liste_compte
                            if c.account_name == account_name), None)
            if account is None:
                continue

            for tid in tids:
                row = account.df[account.df["real_index"] == tid]
                if row.empty:
                    continue

                has_any_tx = True
                tx_date = pd.to_datetime(
                    row.iloc[0].get(COLUMNS_STRUCTURE[0], ""),
                    dayfirst=True, errors="coerce"
                )
                if pd.isna(tx_date):
                    continue

                if tx_date.timestamp() >= cutoff:
                    nb_tx_recent += 1
                    total_recent += float(row.iloc[0].get(COLUMNS_STRUCTURE[5], 0))

        if not has_any_tx:
            return 1.0

        ratio_recent = min(total_recent / step.target, 1.0) if step.target else 0
        bonus = 0.5 * ratio_recent
        malus = 0.5 if nb_tx_recent == 0 else 0.0

        return 1.0 + bonus - malus

    def next_step_to_fund(self, window_months: int = 3) -> Optional[Step]:
        """Étape la plus urgente à financer, tous objectifs confondus."""
        candidates = [
            (self._compute_priority_score(obj, step, window_months), step)
            for obj in self.objectifs
            for step in obj.steps
            if step.status == EN_COURS and step.remaining_to_find > 0
        ]
        if not candidates:
            return None
        return max(candidates, key=lambda x: x[0])[1]

    def ranked_objectives(self) -> list[Objective]:
        """Importance desc, puis progression asc (les plus en retard en premier)."""
        return sorted(self.objectifs,
                      key=lambda o: (-o.importance, o.progress_percent))

    def priority_report(self, window_months: int = 3) -> list[dict]:
        """Tableau de bord complet des scores pour tous les objectifs EN_COURS."""
        report = []
        for obj in self.objectifs:
            for step in obj.steps:
                if step.status != EN_COURS:
                    continue
                score = self._compute_priority_score(obj, step, window_months)
                report.append({
                    "objectif"   : obj.name,
                    "step"       : step.name,
                    "score"      : round(score, 3),
                    "progression": f"{step.progress_percent}%",
                    "restant"    : step.remaining_to_find,
                    "deadline"   : step.deadline or obj.deadline or "—",
                })
        report.sort(key=lambda x: x["score"], reverse=True)
        return report

    # ── Résumé global ─────────────────────────────────────────────────────────

    def global_summary(self) -> dict:
        return {
            "nb_objectifs"  : len(self.objectifs),
            "épargné_total" : sum(o.total_saved    for o in self.objectifs),
            "dépensé_total" : sum(o.total_expenses for o in self.objectifs),
            "objectif_total": sum(o.total_target   for o in self.objectifs),
            "restant_total" : sum(
                max(o.total_target - o.total_saved, 0) for o in self.objectifs
            ),
        }

# Cerveau de la classe

In [9]:
from core.persistence.database_manager import DatabaseManager


# ══════════════════════════════════════════════════════════════════════════════
#  SETUP
# ══════════════════════════════════════════════════════════════════════════════

db_manager = DatabaseManager("Compte")
compta     = db_manager.comptabilite
pm         = ProjetManagement(compta, json_path="./_data/goal.json")

compte_test  = compta.liste_compte[0]
account_name = compte_test.account_name
real_ids     = compte_test.df["real_index"].dropna().tolist()

print(f"✅ Compte de test    : '{account_name}' — {len(real_ids)} transactions")
print(f"✅ Objectifs chargés : {[o.name for o in pm.objectifs]}\n")

# ══════════════════════════════════════════════════════════════════════════════
#  TEST 1 — Résumé global initial
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST 1 — Résumé global initial")
print("=" * 60)
summary = pm.global_summary()
for k, v in summary.items():
    print(f"  {k} : {v}")
assert summary["épargné_total"] == 0, "❌ Épargne devrait être 0"
assert summary["dépensé_total"] == 0, "❌ Dépenses devraient être 0"
print("  ✅ OK\n")

# ══════════════════════════════════════════════════════════════════════════════
#  TEST 2 — Création objectif + étape avec max_keywords custom
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST 2 — Création objectif + étape (max_keywords=5)")
print("=" * 60)
pm.create_objective(
    name       = "Nouvelle voiture",
    but        = "Apport pour un véhicule d'occasion",
    importance = 4,
    deadline   = "2027-06-01",
    logo       = "🚗"
)
pm.create_step(
    obj_name     = "Nouvelle voiture",
    name         = "Apport initial",
    target       = 3000.0,
    but          = "20% du prix du véhicule",
    keywords     = ["voiture", "auto"],
    max_keywords = 5
)
obj_voiture = pm.get_objective("Nouvelle voiture")
step_voiture = obj_voiture.steps[0]
assert obj_voiture  is not None,         "❌ Objectif non trouvé"
assert len(obj_voiture.steps) == 1,      "❌ Step non créée"
assert step_voiture.max_keywords == 5,   "❌ max_keywords non appliqué"
print(f"  Objectif '{obj_voiture.name}' — step '{step_voiture.name}'")
print(f"  max_keywords : {step_voiture.max_keywords}")
print(f"  keywords initiaux : {step_voiture.keywords}")
print("  ✅ OK\n")

# ══════════════════════════════════════════════════════════════════════════════
#  TEST 3 — Liaison transaction (revenu)
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST 3 — Liaison transaction (revenu)")
print("=" * 60)
tid_1  = real_ids[0]
result = pm.link_transaction(
    obj_name     = "Fonds d'urgence",
    step_name    = "Tranche 1 — 1 mois",
    account_name = account_name,
    tid          = tid_1,
    is_expense   = False
)
step_t1 = pm._get_step("Fonds d'urgence", "Tranche 1 — 1 mois")
print(f"  Transaction '{tid_1}' liée")
print(f"  Distribué   : {result.attributed}")
print(f"  Progression : {step_t1.progress_percent}% ({step_t1.current_total}€ / {step_t1.target}€)")
print(f"  Overflow    : {result.overflow}€  overflowed={result.overflowed}")
print(f"  Keywords enrichis : {step_t1.keywords}")
assert tid_1 in step_t1.income_links.get(account_name, []), "❌ ID absent des income_links"
assert len(step_t1.keywords) <= step_t1.max_keywords,       "❌ Limite keywords dépassée"
print("  ✅ OK\n")

# ══════════════════════════════════════════════════════════════════════════════
#  TEST 4 — Liaison transaction (dépense)
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST 4 — Liaison transaction (dépense)")
print("=" * 60)
tid_2  = real_ids[1] if len(real_ids) > 1 else real_ids[0]
result = pm.link_transaction(
    obj_name     = "PC Workstation",
    step_name    = "Composants principaux",
    account_name = account_name,
    tid          = tid_2,
    is_expense   = True
)
step_pc = pm._get_step("PC Workstation", "Composants principaux")
print(f"  Transaction '{tid_2}' liée en dépense")
print(f"  Distribué   : {result.attributed}")
print(f"  Dépensé     : {step_pc.expenses_only}€")
print(f"  Net balance : {step_pc.net_balance}€")
print(f"  Keywords enrichis : {step_pc.keywords}")
assert tid_2 in step_pc.expense_links.get(account_name, []), "❌ ID absent des expense_links"
assert len(step_pc.keywords) <= step_pc.max_keywords,        "❌ Limite keywords dépassée"
print("  ✅ OK\n")

# ══════════════════════════════════════════════════════════════════════════════
#  TEST 5 — Limite max_keywords respectée après enrichissement massif
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST 5 — Respect de la limite max_keywords")
print("=" * 60)
for tid in real_ids[:10]:   # on lie 10 transactions pour saturer les keywords
    pm.link_transaction(
        obj_name     = "Nouvelle voiture",
        step_name    = "Apport initial",
        account_name = account_name,
        tid          = tid,
        is_expense   = False
    )
step_voiture = pm._get_step("Nouvelle voiture", "Apport initial")
print(f"  Keywords après 10 liaisons : {step_voiture.keywords}")
print(f"  Nombre : {len(step_voiture.keywords)} / max {step_voiture.max_keywords}")
assert len(step_voiture.keywords) <= step_voiture.max_keywords, "❌ Limite keywords dépassée"
print("  ✅ OK\n")

# ══════════════════════════════════════════════════════════════════════════════
#  TEST 6 — Cascade surplus → étape suivante
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST 6 — Cascade surplus → étape suivante")
print("=" * 60)
step_t1               = pm._get_step("Fonds d'urgence", "Tranche 1 — 1 mois")
step_t1.current_total = step_t1.target
step_t1.status        = TERMINE

tid_3  = real_ids[2] if len(real_ids) > 2 else real_ids[0]
result = pm.link_transaction(
    obj_name     = "Fonds d'urgence",
    step_name    = "Tranche 1 — 1 mois",
    account_name = account_name,
    tid          = tid_3,
    is_expense   = False
)
print(f"  Distribué : {result.attributed}")
print(f"  Overflow  : {result.overflow}€  overflowed={result.overflowed}")
if result.attributed:
    print(f"  Cascade vers : {list(result.attributed.keys())}")
print("  ✅ OK\n")

# ══════════════════════════════════════════════════════════════════════════════
#  TEST 7 — Recommandations
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST 7 — Recommandations de transactions")
print("=" * 60)
reco = pm.recommend_transactions(
    obj_name     = "PC Workstation",
    step_name    = "Composants principaux",
    account_name = account_name,
    max_results  = 5
)
print(f"  {len(reco)} recommandation(s)")
for r in reco:
    tx = r["transaction"]
    print(f"    score {r['score']} — {tx.get('Intitule')} | {tx.get('Valeur')}€")
print("  ✅ OK\n")

# ══════════════════════════════════════════════════════════════════════════════
#  TEST 8 — Unlink transaction
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST 8 — Déliaison transaction")
print("=" * 60)
found   = pm.unlink_transaction(tid_1)
step_t1 = pm._get_step("Fonds d'urgence", "Tranche 1 — 1 mois")
print(f"  Déliaison de '{tid_1}' : {'trouvé ✅' if found else '❌ non trouvé'}")
assert tid_1 not in step_t1.income_links.get(account_name, []), "❌ ID toujours présent"
print("  ✅ OK\n")

# ══════════════════════════════════════════════════════════════════════════════
#  TEST 9 — Rapport de priorité
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST 9 — Rapport de priorité (window=3 mois)")
print("=" * 60)
report = pm.priority_report(window_months=3)
for row in report:
    print(f"  [{row['score']:>7}] {row['objectif']:<25} → {row['step']:<30} "
          f"| {row['progression']:>6} | reste {row['restant']:>8}€ | {row['deadline']}")
assert len(report) > 0, "❌ Rapport vide"
print("  ✅ OK\n")

# ══════════════════════════════════════════════════════════════════════════════
#  TEST 10 — Prochaine étape à financer
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST 10 — Prochaine étape à financer")
print("=" * 60)
next_step = pm.next_step_to_fund(window_months=3)
if next_step:
    print(f"  → '{next_step.name}' (manque {next_step.remaining_to_find}€)")
else:
    print("  → Aucune étape EN_COURS avec du restant")
print("  ✅ OK\n")

# ══════════════════════════════════════════════════════════════════════════════
#  TEST 11 — Sauvegarde + rechargement
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("TEST 11 — Sauvegarde et rechargement")
print("=" * 60)
pm.save("./_data/goal2.json")
pm2 = ProjetManagement(compta, json_path="./_data/goal2.json")
assert len(pm2.objectifs) == len(pm.objectifs), "❌ Nombre d'objectifs différent"
for o1, o2 in zip(pm.objectifs, pm2.objectifs):
    assert o1.name == o2.name, f"❌ '{o1.name}' != '{o2.name}'"
    for s1, s2 in zip(o1.steps, o2.steps):
        assert s1.keywords     == s2.keywords,     f"❌ Keywords perdus sur '{s1.name}'"
        assert s1.max_keywords == s2.max_keywords,  f"❌ max_keywords perdu sur '{s1.name}'"
print(f"  {len(pm2.objectifs)} objectifs rechargés — keywords et max_keywords préservés")
print("  ✅ OK\n")

print("=" * 60)
print("🎉 Tous les tests passés")
print("=" * 60)

✅ Compte de test    : 'CH_compte_courant' — 40 transactions
✅ Objectifs chargés : ['Voyage au Japon', 'PC Workstation', "Fonds d'urgence"]

TEST 1 — Résumé global initial
  nb_objectifs : 3
  épargné_total : 0
  dépensé_total : 0
  objectif_total : 9600.0
  restant_total : 9600.0
  ✅ OK

TEST 2 — Création objectif + étape (max_keywords=5)
  Objectif 'Nouvelle voiture' — step 'Apport initial'
  max_keywords : 5
  keywords initiaux : ['voiture', 'auto']
  ✅ OK

TEST 3 — Liaison transaction (revenu)
  Transaction 'ac15142b' liée
  Distribué   : {'Tranche 1 — 1 mois': 38.0}
  Progression : 3.2% (38.0€ / 1200.0€)
  Overflow    : 0.0€  overflowed=False
  Keywords enrichis : ['assurance', 'swisscare', 'essentiel']
  ✅ OK

TEST 4 — Liaison transaction (dépense)
  Transaction 'f6711d4c' liée en dépense
  Distribué   : {'Composants principaux': 62.0}
  Dépensé     : 62.0€
  Net balance : 0.0€
  Keywords enrichis : ['sbb', 'chf', 'voyage']
  ✅ OK

TEST 5 — Respect de la limite max_keywords
  Keyw

C:\Users\loube\AppData\Local\Temp\ipykernel_25996\2386810164.py:303: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  tx_date = pd.to_datetime(


In [ ]:
from __future__ import annotations
import json
import os
import uuid
import pandas as pd
from datetime import datetime, date
from typing import Optional
from core.variables.variable import COLUMNS_STRUCTURE, _STOPWORDS, MAX_KEYWORDS, EN_COURS, TERMINE
from core.models.objectif import Objective
from core.models.step import Step


# ─────────────────────────────────────────────────────────────────────────────
# LinkResult
# ─────────────────────────────────────────────────────────────────────────────

class LinkResult:
    def __init__(self):
        self.attributed : dict[str, float] = {}
        self.overflow   : float = 0.0
        self.overflowed : bool  = False

    def __repr__(self):
        return (f"LinkResult(attributed={self.attributed}, "
                f"overflow={self.overflow}, overflowed={self.overflowed})")


# ─────────────────────────────────────────────────────────────────────────────
# Projects — CRUD pur + persistence
# ─────────────────────────────────────────────────────────────────────────────

class Projects:
    """
    Registre des objectifs et étapes.
    Aucune logique métier — juste du CRUD et de la persistence.
    """

    def __init__(self, json_path: str):
        self.objectifs : list[Objective] = []
        self.json_path = json_path
        self.read_json(json_path)

    # ── Persistence ───────────────────────────────────────────────────────────

    def read_json(self, json_path: str) -> None:
        if not os.path.exists(json_path):
            return
        with open(json_path, encoding="utf-8") as f:
            self.objectifs = [Objective.from_dict(d) for d in json.load(f)]

    def save(self, json_path: str = None) -> None:
        target = json_path or self.json_path
        with open(target, "w", encoding="utf-8") as f:
            json.dump([o.to_dict() for o in self.objectifs],
                      f, ensure_ascii=False, indent=2)

    # ── Objectifs ─────────────────────────────────────────────────────────────

    def create_objective(self, objectif: Objective) -> Objective:
        self.objectifs.append(objectif)
        return objectif

    def delete_objective(self, id_objectif: str) -> bool:
        obj = self.get_objectif(id_objectif)
        if obj is None:
            return False
        self.objectifs.remove(obj)
        return True

    def modify_objective(self, id_objectif: str, **kwargs) -> bool:
        """
        Met à jour les champs d'un objectif.
        Champs modifiables : name, but, importance, deadline, logo
        """
        obj = self.get_objectif(id_objectif)
        if obj is None:
            return False
        for key, value in kwargs.items():
            if hasattr(obj, key):
                setattr(obj, key, value)
        return True

    def get_objectif(self, id_objectif: str) -> Optional[Objective]:
        return next((o for o in self.objectifs if o.id == id_objectif), None)

    # ── Steps ─────────────────────────────────────────────────────────────────

    def create_step(self, id_objectif: str, step: Step) -> Optional[Step]:
        obj = self.get_objectif(id_objectif)
        if obj is None:
            return None
        obj.add_step(step)
        return step

    def delete_step(self, id_objectif: str, id_step: str) -> bool:
        obj = self.get_objectif(id_objectif)
        if obj is None:
            return False
        step = self.get_step(id_objectif, id_step)
        if step is None:
            return False
        obj.steps.remove(step)
        return True

    def modify_step(self, id_objectif: str, id_step: str, **kwargs) -> bool:
        """
        Met à jour les champs d'une étape.
        Champs modifiables : name, but, target, deadline, status, max_keywords
        Note : current_total et expenses_only sont gérés par compute(), pas ici.
        """
        step = self.get_step(id_objectif, id_step)
        if step is None:
            return False
        for key, value in kwargs.items():
            if hasattr(step, key):
                setattr(step, key, value)
        return True

    def get_step(self, id_objectif: str, id_step: str) -> Optional[Step]:
        obj = self.get_objectif(id_objectif)
        if obj is None:
            return None
        return next((s for s in obj.steps if s.id == id_step), None)

    # ── Liens transactions ────────────────────────────────────────────────────

    def link_transaction(self, id_objectif: str, id_step: str,
                         account_name: str, tid: str, depense: bool) -> bool:
        """Enregistre l'ID de transaction dans la step. Pas de calcul."""
        step = self.get_step(id_objectif, id_step)
        if step is None:
            return False
        links = step.expense_links if depense else step.income_links
        links.setdefault(account_name, [])
        if tid not in links[account_name]:
            links[account_name].append(tid)
        return True

    def unlink_transaction(self, id_objectif: str, id_step: str,
                           account_name: str, tid: str, depense: bool) -> bool:
        """Retire l'ID de transaction de la step."""
        step = self.get_step(id_objectif, id_step)
        if step is None:
            return False
        links = step.expense_links if depense else step.income_links
        if account_name in links and tid in links[account_name]:
            links[account_name].remove(tid)
            if not links[account_name]:
                del links[account_name]
            return True
        return False


# ─────────────────────────────────────────────────────────────────────────────
# ProjectManagement — logique métier
# ─────────────────────────────────────────────────────────────────────────────

class ProjectManagement:
    """
    Cerveau analytique.
    A accès à Projects (structure) et compta (données financières).
    """

    def __init__(self, projects: Projects, compta):
        self.projects = projects
        self.compta   = compta

    # ── Helpers privés ────────────────────────────────────────────────────────

    def _get_account(self, account_name: str):
        return next((c for c in self.compta.liste_compte
                     if c.account_name == account_name), None)

    def _get_transaction_amount(self, account_name: str, tid: str) -> Optional[float]:
        """Retourne le montant d'une transaction depuis compta."""
        account = self._get_account(account_name)
        if account is None:
            return None
        row = account.df[account.df["real_index"] == tid]
        if row.empty:
            return None
        return float(row.iloc[0].get(COLUMNS_STRUCTURE[5], 0))

    def _get_transaction(self, account_name: str, tid: str) -> Optional[dict]:
        account = self._get_account(account_name)
        if account is None:
            return None
        row = account.df[account.df["real_index"] == tid]
        return row.iloc[0].to_dict() if not row.empty else None

    # ── Compute ───────────────────────────────────────────────────────────────

    def compute(self) -> None:
        """
        Relit tous les IDs depuis compta et recalcule from scratch :
        - current_total  : somme des revenus liés
        - expenses_only  : somme des dépenses liées
        - status         : TERMINE si current_total >= target
        - overflow       : surplus redistribué vers la prochaine step EN_COURS

        Appelé automatiquement par apply_transaction.
        Peut aussi être appelé manuellement après un import ou une modif externe.
        """
        for obj in self.projects.objectifs:
            overflow = 0.0  # surplus de la step précédente à cascader

            for step in obj.steps:
                # ── Reset ────────────────────────────────────────────────────
                step.current_total = 0.0
                step.expenses_only = 0.0

                # ── Revenus ──────────────────────────────────────────────────
                for account_name, tids in step.income_links.items():
                    for tid in tids:
                        amount = self._get_transaction_amount(account_name, tid)
                        if amount is not None:
                            step.current_total += amount

                # ── Dépenses ─────────────────────────────────────────────────
                for account_name, tids in step.expense_links.items():
                    for tid in tids:
                        amount = self._get_transaction_amount(account_name, tid)
                        if amount is not None:
                            step.expenses_only  += amount
                            step.current_total  += amount

                # ── Absorption du surplus de la step précédente ───────────────
                if overflow > 0 and step.status == EN_COURS:
                    absorbable      = min(overflow, step.remaining_to_find)
                    step.current_total += absorbable
                    overflow        -= absorbable

                # ── Statut ───────────────────────────────────────────────────
                if step.current_total >= step.target and step.target > 0:
                    overflow      += step.current_total - step.target
                    step.current_total = step.target
                    step.status   = TERMINE
                elif step.status == TERMINE and step.current_total < step.target:
                    step.status   = EN_COURS  # rétrogradation si on unlink

    # ── apply_transaction ─────────────────────────────────────────────────────

    def apply_transaction(self, id_objectif: str, id_step: str,
                          account_name: str, tid: str,
                          depense: bool) -> LinkResult:
        """
        Link + compute en une seule opération.
        Retourne un LinkResult avec la distribution finale.
        """
        result = LinkResult()

        # 1. Enregistre le lien
        linked = self.projects.link_transaction(
            id_objectif, id_step, account_name, tid, depense
        )
        if not linked:
            return result

        # 2. Recalcule tout from scratch
        self.compute()

        # 3. Construit le LinkResult depuis l'état post-compute
        obj = self.projects.get_objectif(id_objectif)
        if obj is None:
            return result

        total_remaining = 0.0
        for step in obj.steps:
            if step.current_total > 0:
                result.attributed[step.name] = step.current_total
            total_remaining += step.remaining_to_find

        # Overflow = ce qui dépasse tous les steps de l'objectif
        tx_amount = self._get_transaction_amount(account_name, tid) or 0.0
        total_absorbed = sum(result.attributed.values())
        if tx_amount > total_absorbed:
            result.overflow   = tx_amount - total_absorbed
            result.overflowed = True

        return result

    # ── Recommandation ────────────────────────────────────────────────────────

    def recommendation(self, id_objectif: str, id_step: str,
                       account_name: str,
                       max_results: int = 10) -> list[dict]:
        """
        Pour chaque transaction non liée du compte, calcule un score
        basé sur les keywords de la step.
        Enrichit aussi les keywords depuis les transactions déjà liées.
        """
        step = self.projects.get_step(id_objectif, id_step)
        account = self._get_account(account_name)
        if step is None or account is None:
            return []

        # Enrichissement depuis les transactions déjà liées
        all_links = {**step.income_links, **step.expense_links}
        for acc_name, tids in all_links.items():
            for tid in tids:
                tx = self._get_transaction(acc_name, tid)
                if tx:
                    self._enrich_keywords(step, tx)

        if not step.keywords:
            return []

        keywords = [k.lower() for k in step.keywords]
        already_linked = set(
            tid for links in (step.income_links, step.expense_links)
            for ids in links.values() for tid in ids
        )

        results = []
        for _, row in account.df.iterrows():
            tid = row.get("real_index", "")
            if tid in already_linked:
                continue
            haystack = " ".join([
                str(row.get(COLUMNS_STRUCTURE[1], "")),
                str(row.get(COLUMNS_STRUCTURE[2], "")),
                str(row.get(COLUMNS_STRUCTURE[3], "")),
            ]).lower()
            score = sum(1 for kw in keywords if kw in haystack)
            if score > 0:
                results.append({"score": score, "transaction": row.to_dict()})

        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:max_results]

    def _enrich_keywords(self, step: Step, tx: dict) -> None:
        sources = [tx.get(COLUMNS_STRUCTURE[1], ""),
                   tx.get(COLUMNS_STRUCTURE[2], ""),
                   tx.get(COLUMNS_STRUCTURE[3], "")]
        for raw in sources:
            for word in str(raw).lower().split():
                if len(word) > 2 and word not in _STOPWORDS and word not in step.keywords:
                    step.keywords.append(word)
        if len(step.keywords) > step.max_keywords:
            step.keywords = step.keywords[-step.max_keywords:]

    # ── Priorité ─────────────────────────────────────────────────────────────

    def project_priority(self, window_months: int = 3) -> list[str]:
        """
        Calcule une priorité pour chaque objectif et retourne
        les ids triés du plus urgent au moins urgent.

        score = importance × retard × urgence_deadline × activité
        """
        scores = []
        for obj in self.projects.objectifs:
            obj_score = max(
                (self._step_score(obj, step, window_months)
                 for step in obj.steps if step.status == EN_COURS),
                default=0.0
            )
            scores.append((obj.id, obj_score))

        scores.sort(key=lambda x: x[1], reverse=True)
        return [oid for oid, _ in scores]

    def _step_score(self, obj: Objective, step: Step,
                    window_months: int) -> float:
        retard = 1 - (step.current_total / step.target) if step.target else 0

        urgence      = 1.0
        deadline_str = step.deadline or obj.deadline
        if deadline_str:
            try:
                deadline_date = datetime.strptime(deadline_str, "%Y-%m-%d").date()
                mois_restants = max(
                    (deadline_date - date.today()).days / 30.44, 0.1
                )
                urgence = 1 + (1 / mois_restants)
            except ValueError:
                pass

        return obj.importance * retard * urgence * self._activity(step, window_months)

    def _activity(self, step: Step, window_months: int) -> float:
        cutoff       = datetime.now().timestamp() - (window_months * 30.44 * 86400)
        total_recent = 0.0
        nb_tx_recent = 0
        has_any_tx   = False

        for account_name, tids in {**step.income_links, **step.expense_links}.items():
            account = self._get_account(account_name)
            if account is None:
                continue
            for tid in tids:
                row = account.df[account.df["real_index"] == tid]
                if row.empty:
                    continue
                has_any_tx = True
                tx_date = pd.to_datetime(
                    row.iloc[0].get(COLUMNS_STRUCTURE[0], ""),
                    dayfirst=True, errors="coerce"
                )
                if pd.isna(tx_date):
                    continue
                if tx_date.timestamp() >= cutoff:
                    nb_tx_recent += 1
                    total_recent += float(row.iloc[0].get(COLUMNS_STRUCTURE[5], 0))

        if not has_any_tx:
            return 1.0
        bonus = 0.5 * min(total_recent / step.target, 1.0) if step.target else 0
        malus = 0.5 if nb_tx_recent == 0 else 0.0
        return 1.0 + bonus - malus

    # ── Résumé global ─────────────────────────────────────────────────────────

    def global_summary(self) -> dict:
        objs = self.projects.objectifs
        return {
            "nb_objectifs"  : len(objs),
            "épargné_total" : sum(o.total_saved    for o in objs),
            "dépensé_total" : sum(o.total_expenses for o in objs),
            "objectif_total": sum(o.total_target   for o in objs),
            "restant_total" : sum(max(o.total_target - o.total_saved, 0) for o in objs),
        }